In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pathlib

# List of paths for Mask-HybridGNet results
path = "../Results/FetalHead"
paths = os.listdir(path)
paths = [os.path.join(path, p, 'results.csv') for p in paths if os.path.exists(os.path.join(path, p, "results.csv"))]

# Load test set definitions
images_hc18 = "../Dataset/HC18/Landmarks_3_10/test.txt"
images_psfhs = "../Dataset/PSFHS/Landmarks_3_10/test.txt"
images_jnuifm = "../Dataset/JNU-IFM/Landmarks_3_10/test.txt"

images_hc18 = [line.strip() for line in open(images_hc18).readlines()]
images_psfhs = [line.strip() for line in open(images_psfhs).readlines()]
images_jnuifm = [line.strip() for line in open(images_jnuifm).readlines()]

# Process Mask-HybridGNet results
dfs = []
for path in paths:
    model_name = path.split('/')[-2]
    df = pd.read_csv(path)
    df['dataset'] = df['image'].apply(lambda x: 'HC18' if x in images_hc18 else ('PSFHS' if x in images_psfhs else 'JNU-IFM'))
    df['model'] = model_name
    if model_name == "HC18":
        df = df[df['organ'] != 'Pubic symphysis']
    dfs.append(df)

# Load and process nnUNet results
nnunet_path = "../fetal_head_nnunet_results.csv"
if os.path.exists(nnunet_path):
    nnunet_df = pd.read_csv(nnunet_path)
    
    # Map fold names to training dataset names
    fold_mapping = {
        "HC18": "HC18",
        "PSFHS": "PSFHS",
        "JNU-IFM": "JNU-IFM",
        "All": "Merged"
    }
    
    # Rename columns to match format
    nnunet_df = nnunet_df.rename(columns={'fold': 'model'})
    nnunet_df['model'] = nnunet_df['model'].map(fold_mapping)
    
    # Determine dataset from image path
    def get_dataset_from_path(image_path):
        if "HC" in image_path:
            return "HC18"
        elif len(pathlib.Path(image_path).stem) == 5:
            return "PSFHS"
        else:
            return "JNU-IFM"
    
    nnunet_df['dataset'] = nnunet_df['image'].apply(get_dataset_from_path)
    
    # Rename 'Fetal Head' to match your naming convention
    nnunet_df['organ'] = nnunet_df['organ'].replace({'Fetal Head': 'Fetal head'})
    
    # Add prefix to distinguish nnUNet models
    nnunet_df['model'] = 'nnUNet-' + nnunet_df['model']
    
    dfs.append(nnunet_df)
else:
    print(f"Warning: nnUNet results file not found at {nnunet_path}")

# Concatenate all dataframes
df = pd.concat(dfs, ignore_index=True)

# Generate tables for each organ
for organ in df['organ'].unique():
    print(f"\n{'='*80}")
    print(f"Table for {organ}")
    print(f"{'='*80}\n")
    
    subdf = df[df['organ'] == organ]
    
    # Define model order (put nnUNet models at the end)
    mask_models = [m for m in subdf['model'].unique() if not m.startswith('nnUNet')]
    nnunet_models = [m for m in subdf['model'].unique() if m.startswith('nnUNet')]
    models = sorted(mask_models) + sorted(nnunet_models)
    
    datasets = ['HC18', 'JNU-IFM', 'PSFHS']
    metrics = ['DC', 'HD', 'ASSD', 'N']
    
    # Create MultiIndex columns
    columns = pd.MultiIndex.from_product([datasets, metrics])
    
    # Initialize DataFrame
    result_table = pd.DataFrame(index=models, columns=columns)
    result_table.index.name = "Trained on"
    
    # Fill the table
    for model in models:
        for dataset in datasets:
            filtered_data = subdf[(subdf['model'] == model) & (subdf['dataset'] == dataset)]
            if not filtered_data.empty:
                result_table.loc[model, (dataset, 'DC')] = f"{filtered_data['dc'].mean():.3f}"
                result_table.loc[model, (dataset, 'HD')] = f"{filtered_data['hd'].mean():.3f}"
                result_table.loc[model, (dataset, 'ASSD')] = f"{filtered_data['assd'].mean():.3f}"
                result_table.loc[model, (dataset, 'N')] = filtered_data['dc'].count()
            else:
                result_table.loc[model, (dataset, 'DC')] = "N/A"
                result_table.loc[model, (dataset, 'HD')] = "N/A"
                result_table.loc[model, (dataset, 'ASSD')] = "N/A"
                result_table.loc[model, (dataset, 'N')] = "N/A"
    
    print(result_table.to_string())
    print()
    
    # Also generate LaTeX table
    print("\nLaTeX format:")
    print(result_table.to_latex())


Table for Fetal head

                 HC18                      JNU-IFM                       PSFHS                      
                   DC      HD    ASSD    N      DC      HD    ASSD    N     DC       HD    ASSD    N
Trained on                                                                                          
HC18            0.975  11.375   4.416  101   0.827  97.057  40.554  406  0.802   28.611  11.154  510
JNU-IFM         0.884  50.873  21.854  101   0.931  43.841  16.642  406  0.884   17.408   7.051  510
Merged          0.970  14.942   5.440  101   0.951  31.664  11.695  406  0.924   12.302   4.393  510
PSFHS           0.893  47.447  19.754  101   0.947  33.952  12.462  406  0.913   13.383   5.046  510
nnUNet-HC18     0.978   7.098   2.521  101   0.760  67.911  18.423  406  0.810   74.533  21.664  510
nnUNet-JNU-IFM  0.952  23.149   6.188  101   0.356  60.911  21.089  406  0.710  144.637  29.869  510
nnUNet-Merged   0.976  14.196   2.964  101   0.356  52.423  21.641  

In [20]:
import pandas as pd
import numpy as np
import pathlib

# Load nnUNet results
nnunet_df = pd.read_csv("../fetal_head_nnunet_results.csv")

# Map fold names to match your table
fold_mapping = {
    "HC18": "HC18",
    "PSFHS": "PSFHS",  
    "JNU-IFM": "JNU-IFM",
    "All": "Merged"
}
nnunet_df['model'] = nnunet_df['fold'].map(fold_mapping)

# Determine dataset from image path
def get_dataset(image_path):
    if "HC" in image_path:
        return "HC18"
    elif len(pathlib.Path(image_path).stem) == 5:
        return "PSFHS"
    else:
        return "JNU-IFM"

nnunet_df['dataset'] = nnunet_df['image'].apply(get_dataset)

# Create summary tables for each organ
for organ in ['Fetal Head', 'Pubic Symphysis']:
    print(f"\n=== nnUNet Results for {organ} ===")
    subdf = nnunet_df[nnunet_df['organ'] == organ]
    
    models = ['Merged', 'JNU-IFM', 'HC18', 'PSFHS']
    datasets = ['HC18', 'JNU-IFM', 'PSFHS']
    
    for model in models:
        row_data = []
        for dataset in datasets:
            filtered = subdf[(subdf['model'] == model) & (subdf['dataset'] == dataset)]
            if not filtered.empty:
                dc_mean = filtered['dc'].mean()
                dc_std = filtered['dc'].std()
                hd_mean = filtered['hd'].mean()
                hd_std = filtered['hd'].std()
                assd_mean = filtered['assd'].mean()
                assd_std = filtered['assd'].std()
                n = len(filtered)
                
                print(f"{model} on {dataset}: DC={dc_mean:.3f}±{dc_std:.3f}, HD={hd_mean:.3f}±{hd_std:.3f}, ASSD={assd_mean:.3f}±{assd_std:.3f}, N={n}")
            else:
                print(f"{model} on {dataset}: No data")


=== nnUNet Results for Fetal Head ===
Merged on HC18: DC=0.976±0.015, HD=14.196±17.462, ASSD=2.964±1.817, N=101
Merged on JNU-IFM: DC=0.356±0.433, HD=52.423±54.373, ASSD=21.641±31.063, N=406
Merged on PSFHS: DC=0.804±0.283, HD=69.823±40.984, ASSD=16.293±13.579, N=510
JNU-IFM on HC18: DC=0.952±0.056, HD=23.149±31.781, ASSD=6.188±9.118, N=101
JNU-IFM on JNU-IFM: DC=0.356±0.421, HD=60.911±61.544, ASSD=21.089±23.946, N=406
JNU-IFM on PSFHS: DC=0.710±0.317, HD=144.637±94.138, ASSD=29.869±32.481, N=510
HC18 on HC18: DC=0.978±0.013, HD=7.098±4.422, ASSD=2.521±1.288, N=101
HC18 on JNU-IFM: DC=0.760±0.249, HD=67.911±41.912, ASSD=18.423±13.217, N=406
HC18 on PSFHS: DC=0.810±0.196, HD=74.533±55.954, ASSD=21.664±19.868, N=510
PSFHS on HC18: DC=0.930±0.103, HD=36.740±42.256, ASSD=8.947±12.544, N=101
PSFHS on JNU-IFM: DC=0.810±0.122, HD=70.611±52.239, ASSD=19.211±14.544, N=406
PSFHS on PSFHS: DC=0.872±0.115, HD=79.389±58.428, ASSD=17.683±15.781, N=510

=== nnUNet Results for Pubic Symphysis ===
Mer

In [19]:
nnunet_df.fold.unique()

array(['HC18', 'PSFHS', 'JNU-IFM', 'All'], dtype=object)